In [0]:
spark.conf.set(
    "fs.azure.account.auth.type.testgen2strg.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type.testgen2strg.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id.testgen2strg.dfs.core.windows.net",
    dbutils.secrets.get(scope="testsecretscope", key="appid")
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret.testgen2strg.dfs.core.windows.net",
    dbutils.secrets.get(scope="testsecretscope", key="apppwd")
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.testgen2strg.dfs.core.windows.net",
    "https://login.microsoftonline.com/0c44fd98-6a6c-40f7-8cbe-a03159095bd8/oauth2/v2.0/token"
)

In [0]:
dbutils.widgets.text('tablename','')

In [0]:
sourcename = dbutils.widgets.get('tablename')

print(sourcename)

In [0]:
src_path = f"abfss://global@testgen2strg.dfs.core.windows.net/gold/dim{sourcename}"

dest_path = "dim" + sourcename

print(src_path)
print(dest_path)

In [0]:
df = spark.read.format("csv").option("header", True).load(src_path)

#df.show()

src_count = df.count()
print(src_count)

In [0]:
spark.conf.set(
    "fs.azure.account.key.testgen2strg.dfs.core.windows.net",
    dbutils.secrets.get(
        scope="testsecretscope",
        key="adlskey"
    )

)

In [0]:
df.write \
    .mode("overwrite") \
    .format("com.databricks.spark.sqldw") \
    .option(
        "url",
        dbutils.secrets.get(
            scope="testsecretscope",
            key="synapseconnectionstring"
        )
    ) \
    .option("dbtable", dest_path) \
    .option(
        "tempDir",
        "abfss://global@testgen2strg.dfs.core.windows.net/tmp/synapse"
    ) \
    .option("forwardSparkAzureStorageCredentials", "true") \
    .save()

In [0]:
dbutils.notebook.exit(
    "source count:" + str(src_count) +
    " destination count:" + str(src_count)
)